# 04 — Regression: predict the next-observation log return

Compare Random Forest Regression and SVR with 7 and 15 features on validation data. Then evaluate a 7-feature Random Forest against a mean-return baseline on the held-out test set.

Run Notebook 02 first so the processed datasets exist.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))
from fx_utils import PAIRS, PROCESSED_DIR, RESULTS_DIR, chronological_split, get_7_features

data = {p: pd.read_csv(PROCESSED_DIR / f"{p}_model_data.csv", index_col=0, parse_dates=True) for p in PAIRS}
splits = {p: chronological_split(data[p]) for p in PAIRS}

from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR
from sklearn.metrics import r2_score, mean_squared_error
import matplotlib.pyplot as plt

def make_regressor(name):
    if name == "Random Forest":
        return RandomForestRegressor(n_estimators=1000, max_depth=100, random_state=42, n_jobs=-1)
    return make_pipeline(StandardScaler(), SVR(kernel="rbf"))

def metrics(actual, pred):
    return r2_score(actual, pred), np.sqrt(mean_squared_error(actual, pred))

print("Loaded model datasets:")
for p in PAIRS:
    print(p, data[p].shape, data[p].index.min(), "to", data[p].index.max())

## 1. Validation: compare models and feature sets

The validation set is for comparing configurations. The test set remains untouched until the final evaluation.

In [ ]:
rows = []
for pair in PAIRS:
    train, val = splits[pair]["train"], splits[pair]["val"]
    target = f"{pair}_target_return"
    f7 = get_7_features(pair)
    f15 = [c for c in train.columns if c.startswith(pair + "_")
           and c not in [target, f"{pair}_target_direction"]]
    for count, cols in [(7, f7), (15, f15)]:
        for name in ["Random Forest", "SVR-RBF"]:
            model = make_regressor(name)
            model.fit(train[cols], train[target])
            pred = model.predict(val[cols])
            r2, rmse = metrics(val[target], pred)
            base_pred = np.full(len(val), train[target].mean())
            base_r2, base_rmse = metrics(val[target], base_pred)
            rows.append({"Currency Pair":pair, "Model":name, "Features":count,
                         "Validation R²":r2, "Validation RMSE":rmse,
                         "Baseline R²":base_r2, "Baseline RMSE":base_rmse})
regression_validation_results = pd.DataFrame(rows)
display(regression_validation_results.round(6))

## 2. Final test: 7-feature Random Forest versus baseline

This is a reference evaluation, not an assumption that Random Forest is successful. Earlier validation results showed weak regression performance.

In [ ]:
test_rows = []
for pair in PAIRS:
    train, val, test = splits[pair]["train"], splits[pair]["val"], splits[pair]["test"]
    development = pd.concat([train, val])
    target = f"{pair}_target_return"
    cols = get_7_features(pair)
    model = make_regressor("Random Forest")
    model.fit(development[cols], development[target])
    pred = model.predict(test[cols])
    base_pred = np.full(len(test), development[target].mean())
    r2, rmse = metrics(test[target], pred)
    base_r2, base_rmse = metrics(test[target], base_pred)
    test_rows.append({"Currency Pair":pair, "Model":"Random Forest", "Features":7,
                      "Test R²":r2, "Test RMSE":rmse,
                      "Baseline R²":base_r2, "Baseline RMSE":base_rmse})
regression_test_results = pd.DataFrame(test_rows)
display(regression_test_results.round(6))

In [ ]:
# Diagnostic plot for GBP/USD; not a profitability chart.
pair = "GBP_USD"
train, val, test = splits[pair]["train"], splits[pair]["val"], splits[pair]["test"]
dev = pd.concat([train, val])
target = f"{pair}_target_return"
cols = get_7_features(pair)
model = make_regressor("Random Forest").fit(dev[cols], dev[target])
pred = model.predict(test[cols])

plt.figure(figsize=(12, 5))
plt.plot(test.index, test[target], label="Actual")
plt.plot(test.index, pred, label="Predicted")
plt.title("GBP/USD: actual vs predicted next-observation log return")
plt.xlabel("Date")
plt.ylabel("Log return")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
regression_validation_results.to_csv(RESULTS_DIR / "regression_validation_results.csv", index=False)
regression_test_results.to_csv(RESULTS_DIR / "regression_test_results.csv", index=False)
print("Saved regression result tables to", RESULTS_DIR)

## Interpretation

- Negative R² means the model is worse than a constant-mean reference on that evaluation period.
- Compare RMSE with the baseline RMSE; lower is better.
- A predicted log return is not automatically a trading profit. A trading strategy needs explicit entry/exit rules, costs and a separate backtest.